<a href="https://colab.research.google.com/github/kavyashrees1204-hub/todo_api/blob/main/day-03%20aiml/loan_approval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
df=pd.read_csv("/content/Loan-Approval-Prediction.csv")

In [5]:
print(df.head())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

In [4]:
X=df.drop("Loan_Status",axis=1)
Y=df["Loan_Status"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [8]:
# Fill missing values for numerical columns
df["LoanAmount"].fillna(df["LoanAmount"].median(), inplace=True)
df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].mode()[0], inplace=True)
df["Credit_History"].fillna(df["Credit_History"].mode()[0], inplace=True)

# Encode target column (Y = 1, N = 0)
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

# Convert categorical variables to one-hot encoded variables
df_encoded = pd.get_dummies(df.drop("Loan_ID", axis=1), drop_first=True)

# Re-split features and target
X = df_encoded.drop("Loan_Status", axis=1)
y = df_encoded["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

/tmp/ipykernel_447/3819360294.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["LoanAmount"].fillna(df["LoanAmount"].median(), inplace=True)
/tmp/ipykernel_447/3819360294.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
# Define model structure
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train the model
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5077 - loss: 0.6922 - val_accuracy: 0.5758 - val_loss: 0.6693
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6097 - loss: 0.6582 - val_accuracy: 0.5960 - val_loss: 0.6479
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6633 - loss: 0.6349 - val_accuracy: 0.6162 - val_loss: 0.6343
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6837 - loss: 0.6178 - val_accuracy: 0.6869 - val_loss: 0.6212
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7066 - loss: 0.6034 - val_accuracy: 0.6970 - val_loss: 0.6130
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7270 - loss: 0.5901 - val_accuracy: 0.7172 - val_loss: 0.6075
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7372 - loss: 0.5779 - val_accuracy: 0.7273 - val_loss: 0.6023
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7474 - loss: 0.5654 - val_accuracy: 0.7374 - val_loss

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Test data accuracy evaluation
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%\n")

# 2. Predictions generate panni 0 or 1-a convert panrom
y_pred_probs = model.predict(X_test_scaled)
y_pred = (y_pred_probs > 0.5).astype(int)

# 3. Confusion Matrix and Classification Report print panrom
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8374 - loss: 0.4511
Test Accuracy: 83.74%

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Confusion Matrix:
[[24 14]
 [ 6 79]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.63      0.71        38
           1       0.85      0.93      0.89        85

    accuracy                           0.84       123
   macro avg       0.82      0.78      0.80       123
weighted avg       0.83      0.84      0.83       123



In [12]:
# 1. Model-a H5 format-la save panrom
model.save("loan_approval_model.h5")
print("Model successfully saved!")

# 2. Example: Single new applicant prediction test
# (X_test_scaled-lirundhu oru sample vachi predict panni paakrom)
sample_applicant = X_test_scaled[0].reshape(1, -1)
prediction_prob = model.predict(sample_applicant)[0][0]

if prediction_prob > 0.5:
    print(f"Loan Approved! (Confidence: {prediction_prob*100:.2f}%)")
else:
    print(f"Loan Rejected! (Confidence: {(1-prediction_prob)*100:.2f}%)")

Model successfully saved!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Loan Rejected! (Confidence: 93.00%)


In [15]:
!git config --global user.name "kavyashrees1204-hub"
!git config --global user.email "kavyashrees1209@gmail.com"

# GitHub token & repo details replace pannunga
!git clone https://YOUR_PERSONAL_ACCESS_TOKEN@github.com/kavyashrees1204-hub/YOUR_REPO_NAME.git

%cd YOUR_REPO_NAME
!cp /content/Untitled2.ipynb .
!cp /content/loan_approval_model.h5 .

!git add .
!git commit -m "Add trained Loan Approval Model and Notebook"
!git push origin main

Cloning into 'YOUR_REPO_NAME'...
fatal: could not read Password for 'https://YOUR_PERSONAL_ACCESS_TOKEN@github.com': No such device or address
[Errno 2] No such file or directory: 'YOUR_REPO_NAME'
/content
cp: cannot stat '/content/Untitled2.ipynb': No such file or directory
cp: '/content/loan_approval_model.h5' and './loan_approval_model.h5' are the same file
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
